# Arricchimento comuni / province / regioni

Notebook per arricchire i `soggetti` del JSON con dati territoriali a partire da `cls_comuni.csv`.

## Cosa fa
- usa il campo `comune` se presente
- se `comune` è vuoto, prova `comune_runts`
- se ancora vuoto, prova a estrarre il comune dal `nome` dell'ente
- confronta con `cls_comuni.csv`
- aggiunge:
  - `comune_matchato`
  - `codice_comune`
  - `codice_provincia`
  - `provincia`
  - `sigla_provincia`
  - `codice_regione`
  - `regione`
  - `match_comune`
  - `comune_fonte`

## Input attesi
- `08_risultati_enriched_2.2_enti_sovracomunali.json`
- `cls_comuni.csv`


In [1]:
# === CONFIG ===

import os
import json
import copy
import re
import unicodedata
from pathlib import Path

import pandas as pd


CLASSIFICATION_FOLDER = r"classification/"
COMUNI_CSV = os.path.join(CLASSIFICATION_FOLDER, "cls_elenco_comuni_2026.csv")
PROVINCE_CSV = os.path.join(CLASSIFICATION_FOLDER, "cls_elenco_province_2026.csv")
REGIONI_CSV = os.path.join(CLASSIFICATION_FOLDER, "cls_elenco_regioni_2026.csv")
RUNTS_FILE = os.path.join(CLASSIFICATION_FOLDER, "cls_runts.csv")
JSON_STEP1_FOLDER = r"output/json/step_1/"

FUZZY_THRESHOLD = 95

from IPython.display import display


In [2]:


try:
    from rapidfuzz import process, fuzz
    RAPIDFUZZ_OK = True
except Exception:
    RAPIDFUZZ_OK = False

print("rapidfuzz disponibile:", RAPIDFUZZ_OK)


rapidfuzz disponibile: True


In [3]:
def load_province_regioni(PROVINCE_CSV, REGIONI_CSV):
    df_province = pd.read_csv(PROVINCE_CSV, dtype=str, sep=";").fillna("")
    df_regioni = pd.read_csv(REGIONI_CSV, dtype=str, sep=";").fillna("")

    df_province = df_province.rename(columns={
        "Provincia/Uts": "provincia",
        "Regione": "regione",
        "Codice Provincia/Uts": "codice_provincia",
        "Sigla automobilistica": "sigla_provincia",
        "Codice Regione": "codice_regione",
    })

    df_regioni = df_regioni.rename(columns={
        "Regione": "regione",
        "Codice Regione": "codice_regione"
    })

    required_prov = ["provincia", "regione", "codice_provincia", "sigla_provincia", "codice_regione"]
    missing_prov = [c for c in required_prov if c not in df_province.columns]
    if missing_prov:
        raise ValueError(f"Colonne mancanti nel CSV province: {missing_prov}")

    required_reg = ["regione", "codice_regione"]
    missing_reg = [c for c in required_reg if c not in df_regioni.columns]
    if missing_reg:
        raise ValueError(f"Colonne mancanti nel CSV regioni: {missing_reg}")

    def norm_text(s):
        if not s:
            return ""
        s = str(s).strip().upper()
        s = "".join(
            ch for ch in unicodedata.normalize("NFKD", s)
            if not unicodedata.combining(ch)
        )
        s = s.replace("’", "'").replace("‘", "'").replace("`", "'")
        s = re.sub(r"\s+", " ", s).strip()
        return s

    df_province["provincia_norm"] = df_province["provincia"].apply(norm_text)
    df_province["regione_norm"] = df_province["regione"].apply(norm_text)
    df_regioni["regione_norm"] = df_regioni["regione"].apply(norm_text)

    print("✅ Province caricate:", df_province.shape)
    print("✅ Regioni caricate:", df_regioni.shape)

    return df_province, df_regioni


In [4]:
df_province, df_regioni = load_province_regioni(
    PROVINCE_CSV,
    REGIONI_CSV
)

print(df_province.head())
print(df_regioni.head())

print(df_province.columns.tolist())
print(df_regioni.columns.tolist())


✅ Province caricate: (110, 14)
✅ Regioni caricate: (20, 9)
  Codice Ripartizione geografica codice_regione Codice Provincia (storico)  \
0                              1             01                        002   
1                              1             01                        003   
2                              1             01                        004   
3                              1             01                        005   
4                              1             01                        006   

  codice_provincia    provincia Ripartizione geografica   regione  \
0              002     Vercelli              Nord-ovest  Piemonte   
1              003       Novara              Nord-ovest  Piemonte   
2              004        Cuneo              Nord-ovest  Piemonte   
3              005         Asti              Nord-ovest  Piemonte   
4              006  Alessandria              Nord-ovest  Piemonte   

  Tipologia Provincia/Uts Descrizione tipologia Provincia

In [5]:

def load_comuni_csv(csv_path: str) -> pd.DataFrame:
    """Carica il CSV ISTAT dei comuni con fallback di encoding/separatore."""
    attempts = [
        {"encoding": "utf-8-sig", "sep": ";"},
        {"encoding": "utf-8", "sep": ";"},
        {"encoding": "latin1", "sep": ";"},
        {"encoding": "cp1252", "sep": ";"},
        {"encoding": "latin1", "sep": ","},
        {"encoding": "cp1252", "sep": ","},
    ]
    last_error = None
    for kw in attempts:
        try:
            df = pd.read_csv(csv_path, dtype=str, **kw).fillna("")
            if len(df.columns) >= 5:
                print(f"CSV caricato con encoding={kw['encoding']} sep={kw['sep']!r}")
                return df
        except Exception as e:
            last_error = e
    raise RuntimeError(f"Impossibile leggere {csv_path}: {last_error}")


In [6]:

df_comuni = load_comuni_csv(COMUNI_CSV)
print("Righe:", len(df_comuni))
print("Colonne:")
for c in df_comuni.columns:
    print("-", c)
display(df_comuni.head(3))


CSV caricato con encoding=utf-8-sig sep=';'
Righe: 7894
Colonne:
- Codice Ripartizione geografica
- Codice Regione
- Codice Provincia (Storico)
- Codice Provincia/Uts
- Codice Comune (alfanumerico)
- Codice Comune (numerico)
- Comune
- Comune (dizione italiana)
- Comune (dizione straniera)
- Ripartizione geografica
- Regione
- Provincia/Uts
- Flag tipo Uts
- Capoluogo di regione
- Capoluogo di provincia/uts
- Sigla automobilistica
- Codice catasto
- Codice fiscale
- Codice NUTS1 2024
- Codice NUTS2 2024
- Codice NUTS3 2024


,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione italiana),Comune (dizione straniera),Ripartizione geografica,...,Provincia/Uts,Flag tipo Uts,Capoluogo di regione,Capoluogo di provincia/uts,Sigla automobilistica,Codice catasto,Codice fiscale,Codice NUTS1 2024,Codice NUTS2 2024,Codice NUTS3 2024
0,1,01,001,201,001001,1001,Agliè,Agliè,,Nord-ovest,...,Torino,3,0,0,TO,A074,83501790014,ITC,ITC1,ITC11
1,1,01,001,201,001002,1002,Airasca,Airasca,,Nord-ovest,...,Torino,3,0,0,TO,A109,85002910017,ITC,ITC1,ITC11
2,1,01,001,201,001003,1003,Ala di Stura,Ala di Stura,,Nord-ovest,...,Torino,3,0,0,TO,A117,83002970016,ITC,ITC1,ITC11


In [7]:
#old
def _strip_accents(s: str) -> str:
    if not s:
        return ""
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", str(s))
        if not unicodedata.combining(ch)
    )

def _norm_text(s: str) -> str:
    if not s:
        return ""
    s = str(s).strip().upper()
    s = _strip_accents(s)
    repl = {
        "’": "'",
        "‘": "'",
        "`": "'",
        "“": '"',
        "”": '"',
    }
    for a, b in repl.items():
        s = s.replace(a, b)
    s = re.sub(r"[^A-Z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _compact(s: str) -> str:
    return re.sub(r"[^A-Z0-9]", "", _norm_text(s))

def _first_nonempty(*values):
    for v in values:
        if isinstance(v, str) and v.strip():
            return v.strip()
    return ""

def _extract_comune_from_nome(nome: str) -> str:
    """Estrazione conservativa dal nome dell'ente."""
    if not nome:
        return ""
    nome_clean = str(nome).strip()

    patterns = [
        r"^COMUNE DI\s+(.+)$",
        r"^PREFETTURA DI\s+(.+)$",
        r"^QUESTURA DI\s+(.+)$",
        r"^TRIBUNALE DI\s+(.+)$",
        r"^PROCURA DELLA REPUBBLICA PRESSO IL TRIBUNALE DI\s+(.+)$",
        r"^ORDINE DEGLI AVVOCATI DI\s+(.+)$",
        r"^ORDINE DEI FARMACISTI DI\s+(.+)$",
        r"^ORDINE DEI MEDICI.* DI\s+(.+)$",
        r"^AZIENDA USL DI\s+(.+)$",
        r"^AZIENDA OSPEDALIERA UNIVERSITARIA DI\s+(.+)$",
        r"^UNIVERSITA DEGLI STUDI DI\s+(.+)$",
        r"^UNIVERSITÀ DEGLI STUDI DI\s+(.+)$",
        r"^COMANDO PROVINCIALE DEI CARABINIERI DI\s+(.+)$",
        r"^COMANDO PROVINCIALE DELLA GUARDIA DI FINANZA DI\s+(.+)$",
    ]
    norm = _norm_text(nome_clean)
    for pat in patterns:
        m = re.match(pat, norm)
        if m:
            return m.group(1).strip().title()

    # fallback debole: ultima parte dopo " DI "
    if " DI " in norm:
        tail = norm.split(" DI ")[-1].strip()
        if 2 <= len(tail) <= 40 and " E " not in tail:
            return tail.title()

    return ""


In [8]:
# Mappatura colonne ISTAT dal CSV caricato (file 2026)
COL_COMUNE = "Comune"
COL_COD_COMUNE = "Codice Comune (alfanumerico)"
COL_COD_PROV = "Codice Provincia/Uts"
COL_PROV = "Provincia/Uts"
COL_SIGLA_PROV = "Sigla automobilistica"
COL_COD_REG = "Codice Regione"
COL_REG = "Regione"

required_cols = [COL_COMUNE, COL_COD_COMUNE, COL_COD_PROV, COL_PROV, COL_SIGLA_PROV, COL_COD_REG, COL_REG]
missing = [c for c in required_cols if c not in df_comuni.columns]
if missing:
    raise ValueError(f"Colonne mancanti nel CSV comuni: {missing}")

idx_exact = {}
idx_compact = {}
fuzzy_names = []
fuzzy_rows = []

for _, row in df_comuni.iterrows():
    comune_raw = row[COL_COMUNE]
    comune_norm = _norm_text(comune_raw)
    comune_comp = _compact(comune_raw)
    record = row.to_dict()

    if comune_norm and comune_norm not in idx_exact:
        idx_exact[comune_norm] = record
    if comune_comp and comune_comp not in idx_compact:
        idx_compact[comune_comp] = record

    if comune_norm:
        fuzzy_names.append(comune_norm)
        fuzzy_rows.append(record)

print("Indice exact:", len(idx_exact))
print("Indice compact:", len(idx_compact))
print("Fuzzy names:", len(fuzzy_names))


Indice exact: 7887
Indice compact: 7886
Fuzzy names: 7893


In [9]:
def lookup_comune_smart(raw_value: str, fuzzy_threshold: int = 95):
    if not raw_value:
        return None, "none", 0, ""

    q_exact = _norm_text(raw_value)
    q_comp = _compact(raw_value)

    if q_exact in idx_exact:
        return idx_exact[q_exact], "exact", 100, q_exact

    if q_comp in idx_compact:
        return idx_compact[q_comp], "compact", 100, q_exact

    if RAPIDFUZZ_OK and fuzzy_names:
        match = process.extractOne(q_exact, fuzzy_names, scorer=fuzz.token_sort_ratio)
        if match:
            best_name, score, idx = match
            score = int(score)
            if score >= fuzzy_threshold:
                return fuzzy_rows[idx], "fuzzy", score, best_name

    return None, "none", 0, ""

def enrich_comune_provincia_regione_smart(
    INPUT_JSON: str,
    comuni_csv_path: str,
    OUTPUT_JSON: str,
    fuzzy_threshold: int = 95
):
    # usa il df già caricato se il path coincide
    with open(INPUT_JSON, "r", encoding="utf-8") as f:
        data = json.load(f)

    enriched = copy.deepcopy(data)

    stats = {
        "tot_soggetti": 0,
        "con_match": 0,
        "senza_match": 0,
        "exact": 0,
        "compact": 0,
        "fuzzy": 0,
        "fonte_comune": 0,
        "fonte_comune_runts": 0,
        "fonte_nome": 0,
        "fonte_none": 0,
    }

    for file_item in enriched:
        soggetti = file_item.get("soggetti") or []

        for ent in soggetti:
            stats["tot_soggetti"] += 1

            comune = _first_nonempty(ent.get("comune"))
            comune_runts = _first_nonempty(ent.get("comune_runts"))
            comune_nome = _extract_comune_from_nome(ent.get("nome", ""))

            candidate = ""
            fonte = "none"

            if comune:
                candidate = comune
                fonte = "comune"
                stats["fonte_comune"] += 1
            elif comune_runts:
                candidate = comune_runts
                fonte = "comune_runts"
                stats["fonte_comune_runts"] += 1
            elif comune_nome:
                candidate = comune_nome
                fonte = "nome"
                stats["fonte_nome"] += 1
            else:
                stats["fonte_none"] += 1


            fonte_esistente = (ent.get("fonte_provincia_regione") or "").strip()
            is_sovracomunale = fonte_esistente == "elenchi_sovracomunali"

            ent["comune_fonte"] = fonte if not is_sovracomunale else "not_applicable_sovracomunale"
            ent["comune_matchato"] = ""
            ent["codice_comune"] = ""
            # se la fonte è sovracomunale NON sovrascrivere i codici già valorizzati in s2.2
            if not is_sovracomunale:
                ent["codice_provincia"] = ""
                ent["sigla_provincia"] = ""
                ent["codice_regione"] = ""
            ent["match_comune"] = "none" if not is_sovracomunale else ent.get("match_comune", "not_applicable_sovracomunale")

            if is_sovracomunale:
                # per i sovracomunali i codici provincia/regione arrivano da s2.2 e non vanno toccati qui
                continue

            if not candidate:
                stats["senza_match"] += 1
                continue

            row, how, score, best = lookup_comune_smart(candidate, fuzzy_threshold=fuzzy_threshold)

            ent["match_comune"] = how

            if row:
                ent["comune_matchato"] = row.get(COL_COMUNE, "")
                ent["codice_comune"] = row.get(COL_COD_COMUNE, "")
                ent["codice_provincia"] = row.get(COL_COD_PROV, "")
                ent["provincia"] = row.get(COL_PROV, "")
                ent["sigla_provincia"] = row.get(COL_SIGLA_PROV, "")
                ent["codice_regione"] = row.get(COL_COD_REG, "")
                ent["regione"] = row.get(COL_REG, "")

                stats["con_match"] += 1
                if how in ("exact", "compact", "fuzzy"):
                    stats[how] += 1
            else:
                stats["senza_match"] += 1

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(enriched, f, indent=2, ensure_ascii=False)

    print("✅ Arricchimento completato")
    for k, v in stats.items():
        print(f"{k}: {v}")
    print("📄 Output:", OUTPUT_JSON)

    return enriched, stats


In [10]:
def _strip_accents(s: str) -> str:
    if not s:
        return ""
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", str(s))
        if not unicodedata.combining(ch)
    )

def _norm_geo(s: str) -> str:
    if not s:
        return ""
    s = str(s).strip().upper()
    s = _strip_accents(s)
    s = s.replace("’", "'").replace("‘", "'").replace("`", "'")
    s = re.sub(r"[^A-Z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s



def enrich_codici_per_sovracomunali(ent, df_province, df_regioni):
    """
    Per enti sovracomunali:
    - NON cerca il comune
    - usa provincia e regione già presenti
    - valorizza:
        codice_provincia
        sigla_provincia
        codice_regione
    """
    tipo = (ent.get("tipo") or "").strip()
    provincia = (ent.get("provincia") or "").strip()
    regione = (ent.get("regione") or "").strip()

    tipi_sovracomunali = {
        "Unione dei Comuni",
        "Comunità Montana",
        "Comunità Territoriale",
    }

    if tipo not in tipi_sovracomunali:
        return ent, False

    # inizializza campi
    ent.setdefault("comune_fonte", "none")
    ent.setdefault("comune_matchato", "")
    ent.setdefault("codice_comune", "")
    ent.setdefault("codice_provincia", "")
    ent.setdefault("sigla_provincia", "")
    ent.setdefault("codice_regione", "")
    ent.setdefault("match_comune", "none")

    # per questi enti niente comune
    ent["comune_fonte"] = "not_applicable_sovracomunale"
    ent["comune_matchato"] = ""
    ent["codice_comune"] = ""
    ent["match_comune"] = "not_applicable_sovracomunale"

    provincia_norm = _norm_text(provincia)
    regione_norm = _norm_text(regione)

    # MATCH PROVINCIA
    if provincia_norm:
        mask_prov = df_province["provincia_norm"] == provincia_norm

        if mask_prov.any():
            rowp = df_province.loc[mask_prov].iloc[0]

            ent["codice_provincia"] = str(rowp.get("codice_provincia", "")).strip()
            ent["sigla_provincia"] = str(rowp.get("sigla_provincia", "")).strip()

            # se codice_regione manca, prova a prenderlo già da province
            if not (ent.get("codice_regione") or "").strip():
                ent["codice_regione"] = str(rowp.get("codice_regione", "")).strip()

            # se regione è vuota, la recupera da df_province
            if not regione:
                reg_from_prov = str(rowp.get("regione", "")).strip()
                if reg_from_prov:
                    ent["regione"] = reg_from_prov
                    regione_norm = _norm_text(reg_from_prov)

    # MATCH REGIONE
    if regione_norm and not (ent.get("codice_regione") or "").strip():
        mask_reg = df_regioni["regione_norm"] == regione_norm

        if mask_reg.any():
            rowr = df_regioni.loc[mask_reg].iloc[0]
            ent["codice_regione"] = str(rowr.get("codice_regione", "")).strip()

    return ent, True



def enrich_only_codes_from_prov_reg(
    INPUT_JSON,
    cls_comuni_csv_path,
    OUTPUT_JSON
):
    """
    Lascia invariati 'provincia' e 'regione' se già presenti.
    Aggiunge solo:
    - codice_provincia
    - sigla_provincia
    - codice_regione
    """

    with open(INPUT_JSON, "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.read_csv(cls_comuni_csv_path, dtype=str).fillna("")

    # colonne reali attese dal file ISTAT cls_comuni
    col_cod_prov = "Codice Provincia/Uts"
    col_prov = "Provincia/Uts"
    col_sigla = "Sigla automobilistica"
    col_cod_reg = "Codice Regione"
    col_reg = "Regione"





    # tabella province-regione unica
    geo_df = (
        df[[col_cod_prov, col_prov, col_sigla, col_cod_reg, col_reg]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # indice per (provincia, regione)
    idx_full = {}
    idx_prov = {}
    idx_reg = {}

    for _, row in geo_df.iterrows():
        prov = _norm_geo(row[col_prov])
        reg = _norm_geo(row[col_reg])

        rec = {
            "codice_provincia": row[col_cod_prov],
            "sigla_provincia": row[col_sigla],
            "codice_regione": row[col_cod_reg],
            "provincia": row[col_prov],
            "regione": row[col_reg],
        }

        if prov and reg:
            idx_full[(prov, reg)] = rec

        if prov and prov not in idx_prov:
            idx_prov[prov] = rec

        if reg and reg not in idx_reg:
            idx_reg[reg] = rec

    enriched = copy.deepcopy(data)

    n_total = 0
    n_match_full = 0
    n_match_prov = 0
    n_match_reg = 0

    for file_item in enriched:
        soggetti = file_item.get("soggetti") or []

        for ent in soggetti:
            n_total += 1

            # 1. prova prima il caso sovracomunale
            ent, handled = enrich_codici_per_sovracomunali(ent, df_province, df_regioni)
            if handled:
                continue

            # 2. altrimenti fai la logica standard sui comuni
            comune = (ent.get("comune") or "").strip()
            comune_runts = (ent.get("comune_runts") or "").strip()



            prov_raw = (ent.get("provincia") or "").strip()
            reg_raw = (ent.get("regione") or "").strip()

            # inizializza i campi solo se assenti
            if "codice_provincia" not in ent:
                ent["codice_provincia"] = ""
            if "sigla_provincia" not in ent:
                ent["sigla_provincia"] = ""
            if "codice_regione" not in ent:
                ent["codice_regione"] = ""

            prov = _norm_geo(prov_raw)
            reg = _norm_geo(reg_raw)

            rec = None

            # 1) match migliore: provincia + regione
            if prov and reg and (prov, reg) in idx_full:
                rec = idx_full[(prov, reg)]
                n_match_full += 1

            # 2) fallback: sola provincia
            elif prov and prov in idx_prov:
                rec = idx_prov[prov]
                n_match_prov += 1

            # 3) fallback: sola regione
            elif reg and reg in idx_reg:
                rec = idx_reg[reg]
                n_match_reg += 1

            if rec:
                # NON toccare provincia/regione già presenti
                if not (ent.get("codice_provincia") or "").strip():
                    ent["codice_provincia"] = rec["codice_provincia"]

                if not (ent.get("sigla_provincia") or "").strip():
                    ent["sigla_provincia"] = rec["sigla_provincia"]

                if not (ent.get("codice_regione") or "").strip():
                    ent["codice_regione"] = rec["codice_regione"]

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(enriched, f, indent=2, ensure_ascii=False)

    print("✅ Arricchimento codici completato")
    print("Totale soggetti:", n_total)
    print("Match provincia+regione:", n_match_full)
    print("Match solo provincia:", n_match_prov)
    print("Match solo regione:", n_match_reg)
    print("📄 Output:", OUTPUT_JSON)

In [11]:


reg_code = "09"   # es. "08" Emilia-Romagna

INPUT_JSON = Path(JSON_STEP1_FOLDER) / f"{reg_code}_risultati_enriched_2.2.json"
OUTPUT_JSON = Path(JSON_STEP1_FOLDER) / f"{reg_code}_risultati_enriched_2.3.json"   

print("INPUT_JSON:", INPUT_JSON)
print("OUTPUT_JSON:", OUTPUT_JSON)


enriched_data, stats = enrich_comune_provincia_regione_smart(
   INPUT_JSON=INPUT_JSON,
   comuni_csv_path=COMUNI_CSV,
   OUTPUT_JSON=OUTPUT_JSON,
   fuzzy_threshold=FUZZY_THRESHOLD
)


#enrich_only_codes_from_prov_reg( 
#    INPUT_JSON=INPUT_JSON,
#    cls_comuni_csv_path=COMUNI_CSV,
#   OUTPUT_JSON=OUTPUT_JSON)

INPUT_JSON: output\json\step_1\09_risultati_enriched_2.2.json
OUTPUT_JSON: output\json\step_1\09_risultati_enriched_2.3.json
✅ Arricchimento completato
tot_soggetti: 412
con_match: 228
senza_match: 183
exact: 228
compact: 0
fuzzy: 0
fonte_comune: 210
fonte_comune_runts: 19
fonte_nome: 34
fonte_none: 149
📄 Output: output\json\step_1\09_risultati_enriched_2.3.json


In [12]:

def flatten_soggetti(json_data):
    rows = []
    for item in json_data:
        file_name = item.get("file", "")
        for ent in (item.get("soggetti") or []):
            rows.append({
                "file": file_name,
                "nome": ent.get("nome", ""),
                "tipo": ent.get("tipo", ""),
                "comune": ent.get("comune", ""),
                "comune_runts": ent.get("comune_runts", ""),
                "comune_fonte": ent.get("comune_fonte", ""),
                "comune_matchato": ent.get("comune_matchato", ""),
                "match_comune": ent.get("match_comune", ""),
                "codice_comune": ent.get("codice_comune", ""),
                "codice_provincia": ent.get("codice_provincia", ""),
                "provincia": ent.get("provincia", ""),
                "sigla_provincia": ent.get("sigla_provincia", ""),
                "codice_regione": ent.get("codice_regione", ""),
                "regione": ent.get("regione", ""),
            })
    return pd.DataFrame(rows)

df_soggetti = flatten_soggetti(enriched_data)
display(df_soggetti.head(20))


,file,nome,tipo,comune,comune_runts,comune_fonte,comune_matchato,match_comune,codice_comune,codice_provincia,provincia,sigla_provincia,codice_regione,regione
0,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Comune di Poggibonsi,Comuni,Poggibonsi,,comune,Poggibonsi,exact,052022,052,Siena,SI,09,Toscana
1,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Comune di Colle di Val d’Elsa,Comuni,Colle di Val d’Elsa,,comune,Colle di Val d'Elsa,exact,052012,052,Siena,SI,09,Toscana
2,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Comune di San Gimignano,Comuni,San Gimignano,,comune,San Gimignano,exact,052028,052,Siena,SI,09,Toscana
3,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Comune di Casole d’Elsa,Comuni,Casole d’Elsa,,comune,Casole d'Elsa,exact,052004,052,Siena,SI,09,Toscana
4,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Comune di Radicondoli,Comuni,Radicondoli,,comune,Radicondoli,exact,052025,052,Siena,SI,09,Toscana
5,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Centro Pari Opportunità Valdelsa,Organismi di parità,,,none,,none,,,,,,
6,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Associazione Donne Insieme Valdelsa,CAV/Centri Antiviolenza,,COLLE DI VAL D'ELSA,comune_runts,Colle di Val d'Elsa,exact,052012,052,Siena,SI,09,Toscana
7,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Casa Rifugio D.I.V.E.,Case Rifugio,,,none,,none,,,,,,
8,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Azienda USL Toscana Sud Est,ASL (consultori familiari e altri servizi terr...,,,none,,none,,,,,,
9,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Consultorio Azienda USL Toscana Sud Est Zona V...,ASL (consultori familiari e altri servizi terr...,,,none,,none,,,,,,


In [13]:

print("=== MATCH COMUNE ===")
display(df_soggetti["match_comune"].value_counts(dropna=False).rename_axis("match_comune").reset_index(name="count"))

print("=== FONTE COMUNE ===")
display(df_soggetti["comune_fonte"].value_counts(dropna=False).rename_axis("comune_fonte").reset_index(name="count"))

print("=== PRIME RIGHE SENZA MATCH ===")
display(
    df_soggetti[df_soggetti["match_comune"] == "none"][
        ["file", "nome", "tipo", "comune", "comune_runts", "comune_fonte"]
    ].head(50)
)


=== MATCH COMUNE ===


,match_comune,count
0,exact,228
1,none,183
2,not_applicable_sovracomunale,1


=== FONTE COMUNE ===


,comune_fonte,count
0,comune,210
1,none,148
2,nome,34
3,comune_runts,19
4,not_applicable_sovracomunale,1


=== PRIME RIGHE SENZA MATCH ===


,file,nome,tipo,comune,comune_runts,comune_fonte
5,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Centro Pari Opportunità Valdelsa,Organismi di parità,,,none
7,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Casa Rifugio D.I.V.E.,Case Rifugio,,,none
8,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Azienda USL Toscana Sud Est,ASL (consultori familiari e altri servizi terr...,,,none
9,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Consultorio Azienda USL Toscana Sud Est Zona V...,ASL (consultori familiari e altri servizi terr...,,,none
10,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Servizio di Salute Mentale Azienda USL Toscana...,ASL (consultori familiari e altri servizi terr...,,,none
11,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,U.O.C. di Psicologia Azienda USL Toscana Sud Est,ASL (consultori familiari e altri servizi terr...,,,nome
12,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,U.O Pronto Soccorso Azienda USL 7 – Monoblocco...,"Ospedale (Pronto soccorso, ecc.)",,,none
13,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,Pronto Soccorso di Campostaggia Azienda USL To...,"Ospedale (Pronto soccorso, ecc.)",,,nome
14,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,SEUS - Servizio Emergenza Urgenza Sociale,Altro,,,none
15,09_ALTA VALDELSA Protocollo-violenza-valdelsa-...,UTES - Unità Territoriali Emergenza Sociale,Altro,,,none


In [14]:

# Export di controllo opzionale
csv_out = Path(OUTPUT_JSON).with_suffix("").as_posix() + ".csv"
df_soggetti.to_csv(csv_out, index=False, encoding="utf-8")
print("CSV flat salvato in:", csv_out)


CSV flat salvato in: output/json/step_1/09_risultati_enriched_2.3.csv
